# 05 — Feature Engineering & Performance Normalization
---
**Purpose:** Implement the approved feature engineering architecture. Constructs the causal target residual (`Target_Tyre_Degradation`), builds tyre state transformations, and calculates lagged rolling trends strictly without future leakage.

**Input:** `outputs/tyre_stints.parquet`
**Output:** `outputs/features_engineered.parquet`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join("..", "outputs")
input_path = os.path.join(OUTPUT_DIR, "tyre_stints.parquet")

print("Loading dataset...")
df = pd.read_parquet(input_path)
# Ensure strictly chronologically sorted
df = df.sort_values(['Year', 'Round', 'Driver', 'LapNumber']).reset_index(drop=True)
print(f"Loaded {len(df):,} laps.")

Loading dataset...
Loaded 101,290 laps.


## 1. Performance Normalization (Target Definition)

In [2]:
# 1. Total Race Laps (Deterministic)
total_laps = df.groupby('RaceId')['LapNumber'].max().rename('Total_Race_Laps')
df = df.merge(total_laps, on='RaceId', how='left')

# 2. Fuel Weight Penalty
# ~0.065s per lap of fuel burned
df['Fuel_Weight_Penalty'] = (df['Total_Race_Laps'] - df['LapNumber']) * 0.065

# 3. Circuit Base Pace (Causal Historical Baseline)
races = df[['Year', 'Round', 'RaceId', 'GrandPrix']].drop_duplicates().sort_values(['Year', 'Round'])
historical_pace = {}

for i, row in races.iterrows():
    gp = row['GrandPrix']
    # Look ONLY at past races
    past_races = races[(races['GrandPrix'] == gp) & 
                       ((races['Year'] < row['Year']) | 
                        ((races['Year'] == row['Year']) & (races['Round'] < row['Round'])))]
    
    if len(past_races) > 0:
        past_ids = past_races['RaceId'].tolist()
        past_pace = df[(df['RaceId'].isin(past_ids)) & (df['is_clean_lap']) & (~df['flag_wet_compound'])]['LapTimeSeconds'].min()
        historical_pace[row['RaceId']] = past_pace if pd.notna(past_pace) else 90.0
    else:
        # First time at track in dataset: Bootstrap using only the first 10 laps to prevent full-race leakage
        first_10 = df[(df['RaceId'] == row['RaceId']) & (df['is_clean_lap']) & (~df['flag_wet_compound']) & (df['LapNumber'] <= 10)]
        if len(first_10) > 0:
            historical_pace[row['RaceId']] = first_10['LapTimeSeconds'].min()
        else:
            historical_pace[row['RaceId']] = df[(df['RaceId'] == row['RaceId']) & (df['is_clean_lap'])]['LapTimeSeconds'].min()

df['Circuit_Base_Pace'] = df['RaceId'].map(historical_pace)

# 4. Target Residual
df['Expected_Non_Tyre_Pace'] = df['Circuit_Base_Pace'] + df['Fuel_Weight_Penalty']
df['Target_Tyre_Degradation'] = df['LapTimeSeconds'] - df['Expected_Non_Tyre_Pace']

print("Performance normalization complete. Causal Target_Tyre_Degradation created.")

Performance normalization complete. Causal Target_Tyre_Degradation created.


## 2. Tyre & Race Progress Features

In [3]:
# 1. Non-linear Tyre Age (Cliff Capture)
df['Log_TyreAge'] = np.log1p(df['TyreAge'])

# 2. Race Progress
df['RaceProgressFraction'] = df['LapNumber'] / df['Total_Race_Laps']

# 3. Compound Ordinal Encoding (Soft=1, Medium=2, Hard=3)
compound_map = {'SOFT': 1, 'MEDIUM': 2, 'HARD': 3, 'INTERMEDIATE': -1, 'WET': -2, 'TEST_UNKNOWN': 0}
df['Compound_Encoded'] = df['Compound'].map(compound_map).fillna(0)

print("Tyre and Progress features engineered.")

Tyre and Progress features engineered.


## 3. Causal Rolling Trends & Lagged Performance

In [4]:
def calc_slope(y):
    y_clean = y.dropna()
    if len(y_clean) < 3: return np.nan
    x = np.arange(len(y_clean))
    return np.polyfit(x, y_clean, 1)[0]

# 1. Lagged Residual (t-1)
# We only lag within the same StintId!
df['Lag1_Pace_Residual'] = df.groupby('StintId')['Target_Tyre_Degradation'].shift(1)

# 2. Rolling 3-Lap Trend (t-3 to t-1)
df['Rolling3_Degradation_Trend'] = df.groupby('StintId')['Lag1_Pace_Residual'].transform(lambda x: x.rolling(3).apply(calc_slope))

print("Causal rolling trends (strictly using t-1 lagged data) engineered.")

Causal rolling trends (strictly using t-1 lagged data) engineered.


## 4. Team Historical Performance Context

In [5]:
# Calculate the team's median pace residual in PAST races.
# First, get team median per race
team_race_pace = df[(df['is_clean_lap']) & (~df['flag_wet_compound'])].groupby(['Year', 'Round', 'RaceId', 'CanonicalTeam'])['Target_Tyre_Degradation'].median().reset_index()
team_race_pace = team_race_pace.sort_values(['Year', 'Round'])

# Calculate expanding historical median (shifted to prevent leakage of current race)
team_race_pace['Team_Median_Pace_Lag1'] = team_race_pace.groupby('CanonicalTeam')['Target_Tyre_Degradation'].transform(lambda x: x.expanding().median().shift(1))

# Merge back
df = df.merge(team_race_pace[['RaceId', 'CanonicalTeam', 'Team_Median_Pace_Lag1']], on=['RaceId', 'CanonicalTeam'], how='left')

# For the first race of the dataset where Lag1 is NaN, fill with the global median to prevent NaNs
global_median = df['Target_Tyre_Degradation'].median()
df['Team_Median_Pace_Lag1'] = df['Team_Median_Pace_Lag1'].fillna(global_median)

print("Team historical context (Team_Median_Pace_Lag1) engineered.")

Team historical context (Team_Median_Pace_Lag1) engineered.


## 5. Summary & Save

In [6]:
feature_cols = [
    'TyreAge', 'Log_TyreAge', 'StintLap', 'Compound_Encoded', 
    'RaceProgressFraction', 'TrackTemp', 'AirTemp', 
    'is_safety_car', 'is_vsc', 'Lag1_Pace_Residual', 
    'Rolling3_Degradation_Trend', 'Team_Median_Pace_Lag1'
]

print("\nFeature Missing Value Check:")
for col in feature_cols + ['Target_Tyre_Degradation']:
    print(f"  {col}: {df[col].isna().sum():,} NaNs ({(df[col].isna().sum()/len(df))*100:.1f}%)")

out_path = os.path.join(OUTPUT_DIR, "features_engineered.parquet")
df.to_parquet(out_path, index=False)
print(f"\nSaved feature-engineered dataset to: {out_path} ({(os.path.getsize(out_path)/1e6):.1f} MB)")
print("\n[OK] Notebook 05 Feature Engineering complete.")


Feature Missing Value Check:
  TyreAge: 948 NaNs (0.9%)
  Log_TyreAge: 948 NaNs (0.9%)
  StintLap: 354 NaNs (0.3%)
  Compound_Encoded: 0 NaNs (0.0%)
  RaceProgressFraction: 0 NaNs (0.0%)
  TrackTemp: 0 NaNs (0.0%)
  AirTemp: 0 NaNs (0.0%)
  is_safety_car: 0 NaNs (0.0%)
  is_vsc: 0 NaNs (0.0%)
  Lag1_Pace_Residual: 7,055 NaNs (7.0%)
  Rolling3_Degradation_Trend: 16,961 NaNs (16.7%)
  Team_Median_Pace_Lag1: 0 NaNs (0.0%)
  Target_Tyre_Degradation: 1,971 NaNs (1.9%)



Saved feature-engineered dataset to: ..\outputs\features_engineered.parquet (8.1 MB)

[OK] Notebook 05 Feature Engineering complete.
